In [2]:
import os
import numpy as np
import pandas as pd
from glob import glob   

In [3]:
def log2pd(file):
  df = pd.DataFrame()
  with open(file, "r") as f:
    for line in f:
      if line.startswith(">"):
        label = line[1:-1]
        next_line = f.readline().strip()
        next_next_line = f.readline().strip()
        df = df.append({"id": label, "sequence": next_line, "structure": next_next_line}, ignore_index=True)

  return df


In [4]:
# TODO: copy the last F1 score function from sincfold metrics

MATCHING_BRACKETS = [["(", ")"], ["[", "]"], ["{", "}"], ["<", ">"], ["A", "a"], ["B", "a"]] 

def fold2bp(struc, xop="(", xcl=")"):
    openxs = []
    bps = []
    for i, x in enumerate(struc):
        if x == xop: openxs.append(i)
        elif x == xcl: 
            if len(openxs)>0: bps.append([openxs.pop()+1, i+1])
            else: return False
    return bps

def dot2bp(struc):
    bp = []
    for brackets in MATCHING_BRACKETS:
        bp = bp + fold2bp(struc, brackets[0], brackets[1])
    return list(sorted(bp))

def f1_score(ref_bp, pre_bp):
    if len(ref_bp)==0 and len(pre_bp)==0:
        return 1
    tp1 = 0
    for rbp in ref_bp:
        # add tolerance of +/- 1 position
        if rbp in pre_bp or \
            [rbp[0],  rbp[1]-1] in pre_bp or \
            [rbp[0],  rbp[1]+1] in pre_bp or \
            [rbp[0]+1,rbp[1]]   in pre_bp or \
            [rbp[0]-1,rbp[1]]   in pre_bp:
            tp1 = tp1 + 1
    tp2 = 0
    for pbp in pre_bp:
        if pbp in ref_bp or \
            [pbp[0],  pbp[1]-1] in ref_bp or \
            [pbp[0],  pbp[1]+1] in ref_bp or \
            [pbp[0]+1,pbp[1]] in ref_bp or \
            [pbp[0]-1,pbp[1]] in ref_bp:
            tp2 = tp2 + 1
            
    fn = len(ref_bp) - tp1
    fp = len(pre_bp) - tp1
    
    tpr = pre = f1 = 0.0
    if tp1+fn>0: tpr = tp1/float(tp1+fn)     # sensitivity (=recall =power)
    if tp1+fp>0: pre = tp2/float(tp1+fp)     # precision (=ppv)
    if tpr+pre>0: f1 = 2*pre*tpr/(pre+tpr)   # F1 score
    
    return f1

In [6]:
# all partitions famfold
log_files = sorted(glob("results/famfold231220/test*.log"))

pred_pds = []
for logf in log_files:
    print(logf)
    dfx = log2pd(logf)
    dfx["base_pairs"] = dfx["structure"].apply(dot2bp)
    dfx.to_csv(logf.replace(".log", ".csv"), index=False)


results/famfold231220/test_famfold_0_20231218-095020.log
results/famfold231220/test_famfold_1_20231218-183121.log
results/famfold231220/test_famfold_2_20231218-183132.log
results/famfold231220/test_famfold_3_20231219-003116.log
results/famfold231220/test_famfold_4_20231218-223410.log
results/famfold231220/test_famfold_5_20231219-172035.log
results/famfold231220/test_famfold_6_20231219-104211.log
results/famfold231220/test_famfold_7_20231219-172104.log
results/famfold231220/test_famfold_8_20231219-080243.log


In [26]:
# all partitions kfold
log_files = sorted(glob("results/kfold230423/test*.log"))

pred_pds = []
for logf in log_files:
    print(logf)
    dfx = log2pd(logf)
    dfx["base_pairs"] = dfx["structure"].apply(dot2bp)
    dfx.to_csv(logf.replace(".log", ".csv"), index=False)


results/kfold230423/test_kfold_0_20230422-173829.log
results/kfold230423/test_kfold_1_20230423-103139.log
results/kfold230423/test_kfold_2_20230423-112918.log
results/kfold230423/test_kfold_3_20230423-190926.log
results/kfold230423/test_kfold_4_20230423-200940.log


In [39]:
# bprna TS0
dfx = log2pd("./redfold.0114a_original_github/test_bprna_ts0_20240123-183048_sinc.log")
dfx.to_csv("redfold_bprna_ts0_240124_sinc.csv", index=None)

In [40]:
# bprna new
dfx = log2pd("./redfold.0114a_original_github/test_bprna_new_20240123-183048_sinc.log")
dfx.to_csv("redfold_bprna_new_240124_sinc.csv", index=None)

In [41]:
# 50 epochs

# bprna TS0
dfx = log2pd("results/test_bprna_ts0_20240124-155456_50epochs.log")
dfx.to_csv("redfold_bprna_ts0_240124_50epochs.csv", index=None)

# bprna new
dfx = log2pd("results/test_bprna_new_20240124-155456_50epochs.log")
dfx.to_csv("redfold_bprna_new_240124_50epochs.csv", index=None)

In [3]:
# testing impact of epochs in ArchiveII cluster-fold

# 15 epochs
name = "test_clusterfold_15e_4_20241227-121111"
dfx = log2pd("results/"+name+".log")
dfx.to_csv("results/"+name+".csv", index=None)

# 50 epochs
name = "test_clusterfold_100e_4_20241227-134850"
dfx = log2pd("results/"+name+".log")
dfx.to_csv("results/"+name+".csv", index=None)

In [4]:
# testing impact of epochs in ArchiveII fam-fold

# 15 epochs
name = "test_famfold_15e_0_20241224-174342"
dfx = log2pd("results/"+name+".log")
dfx.to_csv("results/"+name+".csv", index=None)

# 200 epochs
name = "test_famfold_200e_0_20241223-191758"
dfx = log2pd("results/"+name+".log")
dfx.to_csv("results/"+name+".csv", index=None)

In [7]:
name = "test_3_clustering_folds_0_20241227-230228"
dfx = log2pd("results/"+name+".log")
dfx["base_pairs"] = dfx["structure"].apply(dot2bp)
dfx.to_csv("results/"+name+".csv", index=None)


In [8]:
name = "test_3_clustering_folds_1_20241227-230228"
dfx = log2pd("results/"+name+".log")
dfx["base_pairs"] = dfx["structure"].apply(dot2bp)
dfx.to_csv("results/"+name+".csv", index=None)
dfx.to_csv("results/pred.csv", index=None)

In [27]:
name = "test_3_clustering_folds_2_20241227-230228"
dfx = log2pd("results/"+name+".log")
dfx["base_pairs"] = dfx["structure"].apply(dot2bp)
dfx.to_csv("results/"+name+".csv", index=None)
dfx.to_csv("results/pred.csv", index=None)

In [28]:
name = "test_3_clustering_folds_3_20241227-230228"
dfx = log2pd("results/"+name+".log")
dfx["base_pairs"] = dfx["structure"].apply(dot2bp)
dfx.to_csv("results/"+name+".csv", index=None)
dfx.to_csv("results/pred.csv", index=None)

In [29]:
name = "test_3_clustering_folds_4_20241227-230228"
dfx = log2pd("results/"+name+".log")
dfx["base_pairs"] = dfx["structure"].apply(dot2bp)
dfx.to_csv("results/"+name+".csv", index=None)
dfx.to_csv("results/pred.csv", index=None)

In [25]:
# verify if test partition 2 have the same ids in splits and results

splits_file = "/DATA/lncRNA/data/revisiting/1_random_kfolds/ArchiveII_splits_random.csv"
splits = pd.read_csv(splits_file)

test_file = "results/kfold230423/test_kfold_2_20230423-112918.log"
dfx = log2pd(test_file)

id1 = set(dfx["id"])
print(sorted(id1))
id2 = splits.loc[(splits["fold_number"]==2) & (splits["partition"]=="test")].id
print(sorted(id2))

print(set(id1) == set(id2))

['16s_B.subtilis_domain4', '16s_C.reinhardtii.mito_domain3', '16s_C.reinhardtii.mito_domain4', '16s_E.coli_domain4', '16s_G.intestinalis_domain3', '16s_H.sapiens_domain3', '16s_M.polymorpha_domain3', '16s_S.cerevisiae_domain4', '16s_Z.mays_domain3', '23s_E.coli_domain3', '5s_Acanthamoeba-castellanii-1', '5s_Acholeplasma-laidlawii-1', '5s_Acidovorax-facilis-1', '5s_Acinetobacter-sp.-1', '5s_Acinetospora-crinita-1', '5s_Acremonium-chrysogenum-1', '5s_Acremonium-persicinum-3', '5s_Actinokineospora-riparia-1', '5s_Aeromonas-media-1', '5s_Aeromonas-salmonicida-1', '5s_Agrobacterium-tumefaciens-2', '5s_Alcaligenes-faecalis-1', '5s_Amycolatopsis-lactamdurans-1', '5s_Amycolatopsis-orientalis-1', '5s_Andrias-japonicus-1', '5s_Antheraea-pernyi-1', '5s_Anthopleura-japonica-1', '5s_Aquaspirillum-serpens-1', '5s_Artemia-sp.-1', '5s_Arthrobacter-polychromogenes-1', '5s_Ascobolus-immersus-1', '5s_Asellus-aquaticus-1', '5s_Aspergillus-unguis-1', '5s_Asterina-pectinifera-1', '5s_Aurelia_aurita-1', '5s_

In [5]:
# all partitions hlfold
hls = list(range(5, 100, 5))
folds = [0]

for hl in hls:
    for fold in folds:
        # test_5_hl_folds_hl5_0_20241229-151641.log
        logf = f"results/hl250102/test_5_hl_folds_hl{hl}_{fold}_20241229-151641.log"
        print(logf)
        dfx = log2pd(logf)
        dfx["base_pairs"] = dfx["structure"].apply(dot2bp)
        
        resdir = f"results/hlcsv/hl{hl}/" # {fold}/"
        if not os.path.exists(resdir): os.makedirs(resdir)
        dfx.to_csv(f"{resdir}pred.csv", index=False)


results/hl250102/test_5_hl_folds_hl5_0_20241229-151641.log
results/hl250102/test_5_hl_folds_hl10_0_20241229-151641.log
results/hl250102/test_5_hl_folds_hl15_0_20241229-151641.log
results/hl250102/test_5_hl_folds_hl20_0_20241229-151641.log
results/hl250102/test_5_hl_folds_hl25_0_20241229-151641.log
results/hl250102/test_5_hl_folds_hl30_0_20241229-151641.log
results/hl250102/test_5_hl_folds_hl35_0_20241229-151641.log
results/hl250102/test_5_hl_folds_hl40_0_20241229-151641.log
results/hl250102/test_5_hl_folds_hl45_0_20241229-151641.log
results/hl250102/test_5_hl_folds_hl50_0_20241229-151641.log
results/hl250102/test_5_hl_folds_hl55_0_20241229-151641.log
results/hl250102/test_5_hl_folds_hl60_0_20241229-151641.log
results/hl250102/test_5_hl_folds_hl65_0_20241229-151641.log
results/hl250102/test_5_hl_folds_hl70_0_20241229-151641.log
results/hl250102/test_5_hl_folds_hl75_0_20241229-151641.log
results/hl250102/test_5_hl_folds_hl80_0_20241229-151641.log
results/hl250102/test_5_hl_folds_hl85_0_2

In [9]:
# all partitions simfold
# sims = [30]
# folds = [0, 1, 2, 3, 4]
# time = "20241229-172304"
sims = [40, 50, 60, 70, 80, 90]
folds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
time = "20241229-184200"

for sim in sims:
    for fold in folds:
        # test_6_sim_folds_sim30_0_20241229-172304
        logf = f"results/sim250102/test_6_sim_folds_sim{sim}_{fold}_{time}.log"
        print(logf)
        dfx = log2pd(logf)
        dfx["base_pairs"] = dfx["structure"].apply(dot2bp)
        
        resdir = f"results/simcsv/hl{sim}/{fold}/"
        if not os.path.exists(resdir): os.makedirs(resdir)
        dfx.to_csv(f"{resdir}pred.csv", index=False)


results/sim250102/test_6_sim_folds_sim40_0_20241229-184200.log
results/sim250102/test_6_sim_folds_sim40_1_20241229-184200.log
results/sim250102/test_6_sim_folds_sim40_2_20241229-184200.log
results/sim250102/test_6_sim_folds_sim40_3_20241229-184200.log
results/sim250102/test_6_sim_folds_sim40_4_20241229-184200.log
results/sim250102/test_6_sim_folds_sim40_5_20241229-184200.log
results/sim250102/test_6_sim_folds_sim40_6_20241229-184200.log
results/sim250102/test_6_sim_folds_sim40_7_20241229-184200.log
results/sim250102/test_6_sim_folds_sim40_8_20241229-184200.log
results/sim250102/test_6_sim_folds_sim40_9_20241229-184200.log
results/sim250102/test_6_sim_folds_sim50_0_20241229-184200.log
results/sim250102/test_6_sim_folds_sim50_1_20241229-184200.log
results/sim250102/test_6_sim_folds_sim50_2_20241229-184200.log
results/sim250102/test_6_sim_folds_sim50_3_20241229-184200.log
results/sim250102/test_6_sim_folds_sim50_4_20241229-184200.log
results/sim250102/test_6_sim_folds_sim50_5_20241229-184

In [5]:
# (for R1 of RNA-LLM)
name = "test_bprna_TS0_20250109-210448"
dfx = log2pd("results/"+name+".log")
dfx["base_pairs"] = dfx["structure"].apply(dot2bp)
dfx.to_csv("results/"+name+".csv", index=None)
#dfx.to_csv("results/pred.csv", index=None)

In [6]:
# (for R1 of RNA-LLM)
name = "test_bprna_20250109-210448"
dfx = log2pd("results/"+name+".log")
dfx["base_pairs"] = dfx["structure"].apply(dot2bp)
dfx.to_csv("results/"+name+".csv", index=None)

In [7]:
# (for R1 of RNA-LLM)
name = "test_PDB-RNA_20250109-204339"
dfx = log2pd("results/"+name+".log")
dfx["base_pairs"] = dfx["structure"].apply(dot2bp)
dfx.to_csv("results/"+name+".csv", index=None)